# Phase 7: Generation Pipeline (Locked Gemini v3)

**Pipeline**: Vietnamese Financial News RAG System — v3
**Owner**: Member C | **Hardware**: Colab CPU (API calls only)

### What this notebook does
For every item in the **test set** (`ground_truth_final.jsonl`):
1. Retrieve relevant context chunks using the best-performing configuration from Phase 6 (Automated via Max MRR selection).
2. Build a three-component Vietnamese prompt: [System Instruction] + [Context] + [Question]
3. Generate an answer via `generate_with_gemini()` (Locked strictly to Gemini 3.1 Flash Lite API).
4. Save to `evaluation/generation_results_{backend}.parquet`

> **Model Configuration** (Static Production Endpoint):
> | Priority | Backend | Model |
> |----------|---------|-------|
> | 1 (Locked) | Gemini OpenAI-Compatible Endpoint | `gemini-3.1-flash-lite` |

> ⚠️ **DO NOT modify the test set**. `ground_truth_final.jsonl` is the golden source of truth.

## Cell 0 — Environment Setup

In [1]:
import os, sys, subprocess
from pathlib import Path

def is_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

IN_COLAB = is_colab()
print(f"Environment: {'Google Colab' if IN_COLAB else 'Local'}")

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_ROOT = Path('/content/rag-vn-finance')
    if not REPO_ROOT.exists():
        print("Cloning repo and installing dependencies (first run)...")
        subprocess.run(['git', 'clone', 'https://github.com/thong7d/rag-vn-finance.git', str(REPO_ROOT)])
        req_path = REPO_ROOT / 'requirements.txt'
        if req_path.exists():
            os.system(f'pip install -r "{req_path}" -q')
        print("Install complete. Restarting kernel to reload packages...")
        os.kill(os.getpid(), 9)  # Force Colab RAM reload
    else:
        print("Repo already exists. Skipping install.")

    # Load API keys from .env stored on Drive
    from dotenv import load_dotenv
    load_dotenv('/content/drive/MyDrive/rag-vn-finance/.env')
else:
    REPO_ROOT = Path(os.getcwd()).parent if 'notebooks' in os.getcwd() else Path(os.getcwd())

print(f"Project root: {REPO_ROOT}")
assert REPO_ROOT.exists(), f"Project root not found: {REPO_ROOT}"

src_path = str(REPO_ROOT)
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / '.env')

print("\nCell 0 complete.")


Environment: Local
Project root: d:\000MINHTHONG\Junior - Semester II\TDM & A\FinalProject\finance-news\implementation

Cell 0 complete.


## Cell 1 — Imports, Config & API Verification

In [2]:
import json
import time
import pandas as pd
import faiss
import torch
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

from src.utils import load_config, resolve_path, ensure_dir, get_env
from src.indexing import load_bm25_index
from src.retrieval import DenseRetriever, SparseRetriever, HybridRetriever, create_dense_retriever
from src.generation import generate_answer, RAG_SYSTEM_PROMPT

# ── Load config ────────────────────────────────────────────────────────────────
config = load_config()

# ── Generation parameters ──────────────────────────────────────────────────────
# Limit to the evaluation sample size
EVAL_SAMPLE      = config['evaluation']['eval_sample_size']
TOP_K            = config['retrieval']['top_k_hybrid']
CHECKPOINT_EVERY = 20   # Save parquet every N generated answers

# ── Verify Gemini API key ──────────────────────────────────────────────────────
gemini_key = get_env('GEMINI_API_KEY')
if not gemini_key:
    raise ValueError("GEMINI_API_KEY is not set in the environment. Stopping execution immediately.")
print(f"✅ GEMINI_API_KEY found → will use gemini-3.1-flash-lite")

# ── Load embedding model for dense retrieval ───────────────────────────────────
model_name = config['embedding']['model_name']
device = config['embedding'].get('device', 'cpu')
if device == 'auto':
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"\nLoading embedding model: {model_name} on {device}")
embedding_model = SentenceTransformer(model_name, device=device)
print(f"✅ Embedding model loaded on {embedding_model.device}")


d:\000MINHTHONG\Junior - Semester II\TDM & A\FinalProject\finance-news\.venv\Lib\site-packages\threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


✅ GEMINI_API_KEY found → will use gemini-3.1-flash-lite

Loading embedding model: intfloat/multilingual-e5-large on cpu


d:\000MINHTHONG\Junior - Semester II\TDM & A\FinalProject\finance-news\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


✅ Embedding model loaded on cpu


## Cell 2 — Load Immutable Test Set

Loads `qa_pairs_test.parquet` (20% split from Phase 5, **never modify**).
Falls back to `qa_pairs_filtered.parquet` only if test file is absent.

In [4]:
qa_dir  = resolve_path(config['synthetic_qa'], 'output_dir')
qa_path = os.path.join(REPO_ROOT, qa_dir, 'ground_truth_final.jsonl')

if os.path.exists(qa_path):
    df_eval = pd.read_json(qa_path, lines=True)
    print(f"✅ Loaded ENTIRE test set: {len(df_eval)} QA pairs from {qa_path}")
else:
    raise FileNotFoundError(
        f"QA data not found at {qa_path}. Ensure Phase 5 has been completed."
    )

# Resolve answer column name
ANSWER_COL = 'ground_truth' if 'ground_truth' in df_eval.columns else 'answer'
print(f"Ground-truth column: '{ANSWER_COL}'")
df_eval.head(2)


✅ Loaded ENTIRE test set: 150 QA pairs from d:\000MINHTHONG\Junior - Semester II\TDM & A\FinalProject\finance-news\implementation\synthetic_qa\ground_truth_final.jsonl
Ground-truth column: 'ground_truth'


,question,ground_truth,contexts,doc_id
0,PVEP hiện đứng thứ mấy trong nhóm các công ty ...,PVEP hiện đứng thứ tư.,[Title: PVEF nỗ lực hết sức nhưng vẫn rớt top ...,7b9bb80ca8574351
1,Cơ quan nào thực hiện thu hồi giấy phép hoạt đ...,Ngân hàng Nhà nước Việt Nam.,[Title: Sáp nhập Công ty Tài chính TNHH MTV Ca...,980bc0f83376c393


## Cell 3 — Select Best Retrieval Configuration

Reads `evaluation/retrieval_benchmark.csv` from Phase 6 and picks the
configuration with the highest `MRR` score as the single config to use
for generation. This avoids running 9 × 197 = 1,773 LLM calls.

> To override and use a specific config, manually set `BEST_STRATEGY`
> and `BEST_METHOD` in the cell below.

In [5]:
backend = os.environ.get("VECTOR_STORE_BACKEND", "faiss").lower()
eval_dir      = os.path.join(REPO_ROOT, resolve_path(config['evaluation'], 'output_dir'))
benchmark_csv = os.path.join(eval_dir, f'retrieval_benchmark_{backend}.csv')

if not os.path.exists(benchmark_csv):
    benchmark_csv = os.path.join(eval_dir, 'retrieval_benchmark.csv')

if os.path.exists(benchmark_csv):
    df_bench = pd.read_csv(benchmark_csv)
    # Select configuration with the highest MRR score
    best_row = df_bench.loc[df_bench['MRR'].idxmax()]
    BEST_STRATEGY = best_row['Strategy']
    BEST_METHOD   = best_row['Method']
    print(f"✅ Best config from Phase 6:")
    print(f"   Strategy : {BEST_STRATEGY}")
    print(f"   Method   : {BEST_METHOD}")
    print(f"   MRR      : {best_row['MRR']:.4f}")
    print(f"   NDCG@10  : {best_row['NDCG@10']:.4f}")
else:
    # Manual fallback — set based on known Phase 6 results
    BEST_STRATEGY = 'fixed_size'
    BEST_METHOD   = 'Hybrid'
    print(f"⚠️  retrieval_benchmark.csv not found.")
    print(f"   Using manual defaults: strategy={BEST_STRATEGY}, method={BEST_METHOD}")

print(f"\nWill generate answers using: [{BEST_STRATEGY}] + [{BEST_METHOD}]")


✅ Best config from Phase 6:
   Strategy : sentence_aware
   Method   : Hybrid
   MRR      : 0.8196
   NDCG@10  : 0.8186

Will generate answers using: [sentence_aware] + [Hybrid]


## Cell 4 — Load Retrieval Indexes for Best Config

In [6]:
index_base_dir = os.path.join(REPO_ROOT, resolve_path(config['indexing'], 'output_dir'))
bm25_base_dir  = os.path.join(REPO_ROOT, resolve_path(config['indexing'], 'bm25_dir'))

chunk_ids_path  = os.path.join(index_base_dir, BEST_STRATEGY, 'chunk_ids.json')
metadata_path   = os.path.join(index_base_dir, BEST_STRATEGY, 'metadata.parquet')

with open(chunk_ids_path, 'r', encoding='utf-8') as f:
    chunk_ids = json.load(f)

# Load metadata to map chunk_id → text
df_meta = pd.read_parquet(metadata_path)
chunk_text_map  = dict(zip(df_meta['chunk_id'], df_meta['text']))
chunk_title_map = dict(zip(df_meta['chunk_id'], df_meta.get('title', pd.Series(dtype=str))))

backend = os.environ.get("VECTOR_STORE_BACKEND", "faiss").lower()
if backend == "faiss":
    faiss_path = os.path.join(index_base_dir, BEST_STRATEGY, 'index.faiss')
    assert os.path.exists(faiss_path), f"FAISS index not found: {faiss_path}"
    faiss_index = faiss.read_index(faiss_path)
    dense_retriever = create_dense_retriever(
        backend="faiss",
        index=faiss_index,
        chunk_ids=chunk_ids,
        model=embedding_model
    )
    print(f"✅ Dense index loaded (FAISS) — {faiss_index.ntotal:,} vectors")
elif backend == "qdrant":
    from qdrant_client import QdrantClient
    collection_name = f"{config.get('vector_store', {}).get('qdrant', {}).get('collection_name', 'vn_finance')}_{BEST_STRATEGY}"
    qdrant_url = get_env("QDRANT_URL")
    qdrant_api_key = get_env("QDRANT_API_KEY")
    
    if qdrant_url:
        qdrant_client = QdrantClient(url=qdrant_url, api_key=qdrant_api_key)
    else:
        local_path = config.get("vector_store", {}).get("qdrant", {}).get("local_path", "qdrant_data")
        qdrant_dir = os.path.join(REPO_ROOT, local_path, BEST_STRATEGY)
        assert os.path.exists(qdrant_dir), f"Qdrant index folder not found: {qdrant_dir}"
        qdrant_client = QdrantClient(path=qdrant_dir)
        
    dense_retriever = create_dense_retriever(
        backend="qdrant",
        client=qdrant_client,
        collection_name=collection_name,
        chunk_ids=chunk_ids,
        model=embedding_model
    )
    print(f"✅ Dense index loaded (Qdrant) for strategy: {BEST_STRATEGY}")

# ── BM25 index ─────────────────────────────────────────────────────────────────
bm25_index, bm25_chunk_ids = load_bm25_index(bm25_base_dir, BEST_STRATEGY)
sparse_retriever = SparseRetriever(bm25_index, bm25_chunk_ids)
print(f"✅ BM25 index loaded  — {len(bm25_chunk_ids):,} chunks")

# ── Hybrid retriever ───────────────────────────────────────────────────────────
hybrid_retriever = HybridRetriever(
    dense_retriever, sparse_retriever,
    rrf_k=config['retrieval']['rrf_k']
)

# Map method name to retriever instance
retriever_map = {
    'Dense':  dense_retriever,
    'Sparse': sparse_retriever,
    'Hybrid': hybrid_retriever,
}
retriever = retriever_map[BEST_METHOD]
print(f"\n✅ Active retriever: {BEST_METHOD}")


C:\Users\Inspiron\AppData\Local\Temp\ipykernel_9792\2686544413.py:39: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Collection <vn_finance_sentence_aware> contains 64197 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  qdrant_client = QdrantClient(path=qdrant_dir)


✅ Dense index loaded (Qdrant) for strategy: sentence_aware


[2026-07-07 13:39:21] [INFO] src.indexing: [sentence_aware] BM25 index loaded — 64,197 chunks, avgdl=153.4


✅ BM25 index loaded  — 64,197 chunks

✅ Active retriever: Hybrid


## Cell 5 — Generation Loop

Iterates over the evaluation set in batches.
- Retrieves context for each question using the best config
- Builds the three-component Vietnamese prompt
- Calls `generate_with_gemini()` (Locked strictly to Gemini 3.1 Flash Lite API)
- Checkpoints to `generation_results.parquet` every 20 answers


In [ ]:
import os
import time
import json
import logging
import pandas as pd
from tqdm import tqdm
from openai import OpenAI

# ── Setup Logging ──────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

ensure_dir(eval_dir)
backend = os.environ.get("VECTOR_STORE_BACKEND", "faiss").lower()
out_path = os.path.join(eval_dir, f'generation_results_{backend}.parquet')

# ── Resume from checkpoint if file already exists ─────────────────────────────
if os.path.exists(out_path):
    df_existing = pd.read_parquet(out_path)
    completed_questions = set(df_existing['question'].tolist())
    all_results = df_existing.to_dict('records')
    logger.info(f"🔄 Resuming — {len(all_results)} answers already saved. Skipping completed rows.")
else:
    completed_questions = set()
    all_results = []
    logger.info("🚀 Starting fresh generation run...")

logger.info(f"Total to generate: {len(df_eval) - len(completed_questions)} remaining\n")

# ── Lock Generation to Gemini 3.1 Flash Lite only ──────────────────────────────
gemini_key = get_env('GEMINI_API_KEY')
if not gemini_key:
    logger.error("GEMINI_API_KEY is not set in the environment.")
    raise ValueError("GEMINI_API_KEY is not set in the environment. Stopping execution immediately.")

logger.info("✅ Initializing Gemini API client...")
client = OpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=gemini_key
)
logger.info("✅ Gemini API client loaded successfully.")

GEN_MODEL_NAME = "gemini-3.1-flash-lite"

def generate_with_gemini(question, contexts):
    """
    Call Gemini to generate a RAG answer.
    Returns (answer_str, latency_s, input_tokens, output_tokens, total_tokens).
    Raises on all failures after 3 retries.
    """
    from src.generation import build_rag_prompt, RAG_SYSTEM_PROMPT
    user_prompt = build_rag_prompt(question, contexts)

    for attempt in range(3):
        try:
            t0 = time.perf_counter()
            response = client.chat.completions.create(
                model=GEN_MODEL_NAME,
                messages=[
                    {"role": "system", "content": RAG_SYSTEM_PROMPT},
                    {"role": "user",   "content": user_prompt}
                ],
                temperature=config['generation']['temperature'],
                max_tokens=config['generation'].get('max_tokens', 1024)
            )
            latency_s = time.perf_counter() - t0

            usage = response.usage
            in_tok  = getattr(usage, "prompt_tokens",     0) if usage else 0
            out_tok = getattr(usage, "completion_tokens", 0) if usage else 0
            tot_tok = getattr(usage, "total_tokens",      0) if usage else 0

            answer = response.choices[0].message.content.strip()
            return answer, latency_s, in_tok, out_tok, tot_tok

        except Exception as e:
            err_str = str(e).lower()
            if "429" in err_str or "rate limit" in err_str:
                if attempt < 2:
                    logger.warning(f"[⏳ RATE LIMIT] Waiting 30s before retry (attempt {attempt + 1})...")
                    time.sleep(30)
                    continue
            elif "503" in err_str or "unavailable" in err_str:
                if attempt < 2:
                    logger.warning(f"[⏳ SERVICE BUSY] Waiting 10s before retry (attempt {attempt + 1})...")
                    time.sleep(10)
                    continue
            elif "quota" in err_str or "402" in err_str:
                logger.error("CRITICAL: Quota/funds exhausted. Stopping.")
                raise RuntimeError(f"Quota exceeded: {e}")

            logger.error(f"Gemini API failed: {e}")
            raise e

# ── Main generation loop ───────────────────────────────────────────────────────
logger.info(f"▶️ Starting generation loop for Strategy: {BEST_STRATEGY} | Method: {BEST_METHOD}")

for i, row in tqdm(df_eval.iterrows(), total=len(df_eval), desc="Generating answers"):
    question     = row['question']
    ground_truth = row.get(ANSWER_COL, '')
    doc_id       = row.get('doc_id', '')

    if question in completed_questions:
        continue

    logger.info(f"--- Processing Q: '{question[:60]}...' ---")

    # Step 1: Retrieve relevant context chunks (measure retrieval time)
    logger.info(f"🔍 Retrieving top {TOP_K} contexts using {BEST_METHOD} retriever...")
    t_retrieval_start = time.perf_counter()
    retrieved      = retriever.retrieve(question, top_k=TOP_K)
    retrieval_time_s = time.perf_counter() - t_retrieval_start

    retrieved_ids  = [cid for cid, _ in retrieved]
    contexts       = [
        chunk_text_map.get(cid, '')
        for cid in retrieved_ids
        if cid in chunk_text_map and chunk_text_map.get(cid, '').strip()
    ]
    logger.info(f"✅ Retrieved {len(contexts)} chunks in {retrieval_time_s:.3f}s.")
    retrieved_context = "\n---\n".join(contexts)

    # Step 2: Generate answer + measure metrics
    logger.info(f"🤖 Generating answer with {GEN_MODEL_NAME}...")
    in_tokens, out_tokens, tot_tokens, latency_s, error_msg = 0, 0, 0, 0.0, ""
    try:
        generated_answer, latency_s, in_tokens, out_tokens, tot_tokens = generate_with_gemini(question, contexts)
        throughput_tps = (out_tokens / latency_s) if latency_s > 0 and out_tokens > 0 else 0.0
        logger.info(f"✅ Answer generated. Latency={latency_s:.2f}s | Out={out_tokens} tok | TPS={throughput_tps:.1f}")
    except Exception as e:
        logger.error(f"🛑 CRITICAL ERROR: {e}")
        break  # Halt loop immediately on failure

    # Step 3: Append result (full schema)
    e2e_latency_s = retrieval_time_s + latency_s

    all_results.append({
        'question':            question,
        'ground_truth':        ground_truth,
        'retrieved_context':   retrieved_context,
        'generated_answer':    generated_answer,
        'doc_id':              doc_id,
        'strategy':            BEST_STRATEGY,
        'method':              BEST_METHOD,
        'retrieved_chunk_ids': json.dumps(retrieved_ids, ensure_ascii=False),
        # ── Performance metrics ─────────────────────────────────────────────
        'model_name':          GEN_MODEL_NAME,
        'latency_s':           latency_s,
        'input_tokens':        in_tokens,
        'output_tokens':       out_tokens,
        'total_tokens':        tot_tokens,
        'throughput_tps':      throughput_tps,
        'retrieval_time_s':    retrieval_time_s,
        'e2e_latency_s':       e2e_latency_s,
        'error':               error_msg,
    })
    completed_questions.add(question)

    # Step 4: Checkpoint (atomic save)
    if len(all_results) % CHECKPOINT_EVERY == 0:
        tmp_path = out_path + ".tmp"
        pd.DataFrame(all_results).to_parquet(tmp_path, index=False)
        os.replace(tmp_path, out_path)
        logger.info(f"💾 Checkpoint saved: {len(all_results)} total results.")

    logger.info("⏳ Cooldown 5s before next question...\n")
    time.sleep(5)  # 15 RPM → 4s minimum; using 5s for safety

# Final atomic save
if all_results:
    tmp_path = out_path + ".tmp"
    pd.DataFrame(all_results).to_parquet(tmp_path, index=False)
    os.replace(tmp_path, out_path)
logger.info(f"🎉 Generation complete. {len(all_results)} results → {out_path}")

2026-07-07 14:12:57 [INFO] 🔄 Resuming — 40 answers already saved. Skipping completed rows.
2026-07-07 14:12:57 [INFO] Total to generate: 110 remaining

2026-07-07 14:12:57 [INFO] ✅ Initializing Gemini API client...


2026-07-07 14:12:58 [INFO] ✅ Gemini API client loaded successfully.
2026-07-07 14:12:58 [INFO] ▶️ Starting generation loop for Strategy: sentence_aware | Method: Hybrid
Generating answers:   0%|          | 0/150 [00:00<?, ?it/s]2026-07-07 14:12:58 [INFO] --- Processing Q: 'CTA dự báo yếu tố nào sẽ là động lực quan trọng cho tăng trư...' ---
2026-07-07 14:12:58 [INFO] 🔍 Retrieving top 10 contexts using Hybrid retriever...
2026-07-07 14:13:01 [INFO] ✅ Retrieved 10 chunks successfully.
2026-07-07 14:13:01 [INFO] 🤖 Generating answer with Gemini 3.1 Flash Lite...
2026-07-07 14:13:04 [INFO] HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
2026-07-07 14:13:04 [INFO] ✅ Answer generated successfully (Length: 375 chars).
2026-07-07 14:13:04 [INFO] ⏳ Cooldown 20s before next question...

Generating answers:  27%|██▋       | 41/150 [00:25<01:09,  1.58it/s]2026-07-07 14:13:24 [INFO] --- Processing Q: 'Những tổ chức nào ủng hộ việc tiêm nh

## Cell 6 — Save Final Results & Verify Schema

In [ ]:
# Final save
df_results = pd.DataFrame(all_results)
backend = os.environ.get("VECTOR_STORE_BACKEND", "faiss").lower()
out_path = os.path.join(eval_dir, f'generation_results_{backend}.parquet')
df_results.to_parquet(out_path, index=False)
print(f"✅ Saved {len(df_results)} generation results → {out_path}")

# ── Verify required output schema ─────────────────────────────────────────────
required_cols = [
    'question', 'ground_truth', 'retrieved_context', 'generated_answer',
    'model_name', 'latency_s', 'input_tokens', 'output_tokens', 'total_tokens',
    'throughput_tps', 'retrieval_time_s', 'e2e_latency_s', 'error'
]
missing = [c for c in required_cols if c not in df_results.columns]
assert not missing, f"Missing required columns: {missing}"
print(f"\n✅ Schema verified. Columns: {df_results.columns.tolist()}")

# ── Summary stats ──────────────────────────────────────────────────────────────
n_errors     = df_results['error'].notna() & (df_results['error'] != '')
n_successful = len(df_results) - n_errors.sum()
avg_latency  = df_results['latency_s'].mean()
avg_tps      = df_results['throughput_tps'].mean()
avg_out_tok  = df_results['output_tokens'].mean()
print(f"\nGeneration summary:")
print(f"  Total          : {len(df_results)}")
print(f"  Successful     : {n_successful}")
print(f"  Errors         : {n_errors.sum()}")
print(f"  Strategy       : {BEST_STRATEGY}")
print(f"  Method         : {BEST_METHOD}")
print(f"  Avg Latency    : {avg_latency:.2f}s")
print(f"  Avg Throughput : {avg_tps:.1f} tok/s")
print(f"  Avg Out Tokens : {avg_out_tok:.0f}")

# Preview
df_results[['question', 'generated_answer', 'ground_truth']].head(3)

✅ Saved 150 generation results → d:\000MINHTHONG\Junior - Semester II\TDM & A\FinalProject\finance-news\implementation\evaluation\generation_results_qdrant.parquet

✅ Schema verified. Columns: ['question', 'ground_truth', 'retrieved_context', 'generated_answer', 'doc_id', 'strategy', 'method', 'retrieved_chunk_ids']

Generation summary:
  Total       : 150
  Successful  : 150
  Errors      : 0
  Strategy    : sentence_aware
  Method      : Hybrid


,question,generated_answer,ground_truth
0,PVEP hiện đứng thứ mấy trong nhóm các công ty ...,"Theo thông tin trong ngữ cảnh, PVEP hiện đứng ...",PVEP hiện đứng thứ tư.
1,Cơ quan nào thực hiện thu hồi giấy phép hoạt đ...,Ngân hàng Nhà nước Việt Nam là cơ quan thực hi...,Ngân hàng Nhà nước Việt Nam.
2,Hãng hàng không Emirates sử dụng loại máy bay ...,Để đáp ứng nhu cầu ngày một tăng cao tại Việt ...,Máy bay Boeing 777-300ER


## Cell 7 — Sample Inspection

Display a random sample for quick qualitative review before Phase 8 evaluation.

In [15]:
import random

backend = os.environ.get("VECTOR_STORE_BACKEND", "faiss").lower()
out_path = os.path.join(eval_dir, f'generation_results_{backend}.parquet')
df_results = pd.read_parquet(out_path)

# Print 3 random examples in readable format
sample_indices = random.sample(range(len(df_results)), min(3, len(df_results)))

for idx in sample_indices:
    row = df_results.iloc[idx]
    print(f"{'='*70}")
    print(f"Q  : {row['question']}")
    print(f"GT : {str(row['ground_truth'])[:200]}...")
    print(f"GEN: {str(row['generated_answer'])[:300]}...")
    print()


Q  : Doanh thu và lợi nhuận sau thuế của Tập đoàn Dabaco trong quý II/2020 là bao nhiêu?
GT : Doanh thu đạt 2.290 tỷ đồng, tăng hơn 37% so với cùng kỳ; lợi nhuận sau thuế đạt 401 tỷ đồng, tăng gấp 54 lần cùng kỳ...
GEN: Theo báo cáo tài chính quý II/2020 của Tập đoàn Dabaco Việt Nam, doanh thu của tập đoàn đạt 2.290 tỷ đồng, tăng hơn 37% so với cùng kỳ. Lợi nhuận sau thuế trong quý II/2020 đạt 401 tỷ đồng, tăng gấp 54 lần so với cùng kỳ năm trước....

Q  : Tỷ lệ chuyển đổi cổ phiếu trong thương vụ sáp nhập SouthernBank vào Sacombank là bao nhiêu?
GT : Tỷ lệ chuyển đổi cổ phiếu là 1:0,75, trong đó 1 cổ phiếu SouthernBank được chuyển đổi thành 0,75 cổ phiếu của Sacombank....
GEN: Tỷ lệ chuyển đổi cổ phiếu trong thương vụ sáp nhập SouthernBank vào Sacombank là 1:0,75 (1 cổ phiếu SouthernBank được chuyển đổi thành 0,75 cổ phiếu của Sacombank). Tuy nhiên, tỷ lệ hoán đổi này chưa được Ngân hàng Nhà nước phê duyệt chính thức....

Q  : Khối nhà đầu tư nước ngoài đã bán ròng cổ phiếu STB của Ng